# Exploración y selección de modelo

Este notebook acompaña al pipeline (`python -m src.pipeline`) y al cuadro de mando. Lee los artefactos de `data/publicado/`, comprueba que los indicadores tienen sentido y justifica la elección del modelo.

In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = 'notebook_connected'
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')
PUB = Path('..') / 'data' / 'publicado'
prov = pd.read_parquet(PUB / 'panel_provincias.parquet')
mun = pd.read_parquet(PUB / 'panel_municipios.parquet')
calidad = json.loads((PUB / 'calidad.json').read_text(encoding='utf-8'))
modelo = json.loads((PUB / 'modelo.json').read_text(encoding='utf-8'))
prov.shape, mun.shape, calidad['ultimo_trimestre_precios'], calidad['ultimo_anio_renta']

((6463, 23), (24857, 24), '2026Q2', 2024)

## 1. Calidad de los datos

Cada tabla, cuántas filas pasan las reglas y por qué se apartan las demás.

In [2]:
pd.DataFrame(calidad['validacion']).T[['desde', 'hasta', 'filas', 'validas', 'rechazadas', 'cobertura_pct']]

,desde,hasta,filas,validas,rechazadas,cobertura_pct
precios_prov,1995Q1,2026Q2,6552,6463,89,98.600
precios_mun,2005Q1,2026Q2,24978,24857,121,94.500
renta_prov,2015,2024,509,508,1,97.700
renta_mun,2015,2024,81203,73777,7426,90.800
poblacion_prov,2015,2024,520,520,0,100.000
poblacion_mun,2015,2024,81292,74611,6681,91.700
ipc,2002Q1,2026Q3,5148,5148,0,100.000
alquiler,2011,2024,672,672,0,100.000
gastos_vivienda,2016,2025,190,190,0,100.000
alquiler_prov,2011,2024,681,681,0,93.500


In [3]:
norm = calidad['normalizacion']
print('Municipios emparejados:', norm['municipios_emparejados'], 'de', norm['municipios_fuente_ministerio'])
print('Métodos:', norm['metodos_municipios'])
pd.DataFrame(norm['municipios_en_otra_provincia'])

Municipios emparejados: 306 de 306
Métodos: {'nombre': 297, 'nombre_otra_provincia': 8, 'parecido': 1}


,provincia_fuente,municipio_fuente,cod_mun
0,Santa Cruz de,Adeje,38001
1,Córdoba,Almuñecar,18017
2,Cuenca,Azuqueca de Henares,19046
3,Guadalajara,Illescas,45081
4,Alicante,Almazora/Almassora,12009
5,Alicante,Benicarló,12027
6,Cáceres,Ames,15002
7,Ourense,Cangas,36008


## 2. El índice de esfuerzo hoy

Último trimestre disponible. La renta de los años sin publicar es estimada.

In [4]:
ultimo = prov[prov['trimestre'] == prov['trimestre'].max()]
ultimo.sort_values('esfuerzo', ascending=False)[['provincia', 'precio_m2', 'renta_hogar', 'esfuerzo', 'diferencial', 'criterio_esfuerzo', 'criterio_precio', 'tensionado']].head(10)

,provincia,precio_m2,renta_hogar,esfuerzo,diferencial,criterio_esfuerzo,criterio_precio,tensionado
3648,Málaga,"3,083.500","40,274.318",0.361,41.718,True,True,True
881,"Balears, Illes","3,980.800","53,678.866",0.346,39.188,True,True,True
3522,Madrid,"4,089.600","58,346.301",0.337,31.809,True,True,True
2517,Gipuzkoa,"3,514.600","55,118.198",0.308,5.481,True,True,True
1007,Barcelona,"3,204.200","52,041.629",0.301,12.046,True,True,True
6031,Bizkaia,"3,024.700","52,828.301",0.284,4.165,False,True,True
377,Alicante/Alacant,"2,020.700","37,472.060",0.278,29.494,False,True,True
1385,Cádiz,"2,043.000","38,161.950",0.269,22.532,False,True,True
125,Araba/Álava,"2,531.300","49,447.389",0.262,3.861,False,True,True
4779,Santa Cruz de Tenerife,"2,296.600","42,674.328",0.256,38.695,False,True,True


In [5]:
fig = px.scatter(ultimo, x='diferencial', y='esfuerzo', hover_name='provincia', color='tensionado',
                 color_discrete_map={True: '#D55E00', False: '#0072B2'},
                 labels={'diferencial': 'Precio − IPC a cinco años (puntos)', 'esfuerzo': 'Esfuerzo'})
fig.add_hline(y=0.30, line_dash='dot'); fig.add_vline(x=3, line_dash='dot')
fig.update_yaxes(tickformat='.0%'); fig.update_layout(height=420, template='simple_white')

La mayoría de provincias tensionadas lo están por el **criterio de precios** (cuadrante derecho), no por el de esfuerzo: los precios llevan cinco años creciendo bastante por encima del IPC, pero la cuota sobre la renta solo supera el 30 % en los mercados más caros.

## 3. Evolución: cuántas provincias cumplen cada criterio

In [6]:
evol = (prov[prov['trimestre'] >= '2015Q1'].groupby('trimestre')[['criterio_esfuerzo', 'criterio_precio', 'tensionado']]
        .sum().reset_index().melt('trimestre', var_name='criterio', value_name='provincias'))
fig = px.line(evol, x='trimestre', y='provincias', color='criterio',
              color_discrete_sequence=['#0072B2', '#E69F00', '#D55E00'])
fig.update_layout(height=380, template='simple_white')

## 4. Modelo: ¿supera a las referencias?

Validación temporal: entrenamiento con trimestres anteriores a 2021, prueba con 2021–2022 (etiquetas de 2023–2024, las últimas calculadas con renta publicada). El umbral de decisión de cada modelo se eligió en la validación temporal dentro del entrenamiento, maximizando F2.

In [7]:
def tabla(nivel):
    m = pd.DataFrame(modelo[nivel]['metricas']).T.drop(columns='matriz')
    return m[[c for c in ['recall', 'precision', 'f2', 'roc_auc', 'umbral', 'f2_validacion_temporal'] if c in m]]
print('Provincias · elegido:', modelo['provincias']['elegido'])
tabla('provincias')

Provincias · elegido: Bosque aleatorio


,recall,precision,f2,roc_auc,umbral,f2_validacion_temporal
Referencia: clase mayoritaria,0.000,0.000,0.000,NaN,NaN,NaN
Referencia: persistencia,0.750,0.636,0.724,NaN,NaN,0.791
Regresión logística,0.286,0.941,0.332,0.932,0.850,0.702
Bosque aleatorio,0.696,0.672,0.691,0.938,0.550,0.815
Gradient boosting,0.625,0.583,0.616,0.924,0.700,0.772


In [8]:
print('Municipios · elegido:', modelo['municipios']['elegido'])
tabla('municipios')

Municipios · elegido: Bosque aleatorio


,recall,precision,f2,roc_auc,umbral,f2_validacion_temporal
Referencia: clase mayoritaria,1.000,0.641,0.899,NaN,NaN,NaN
Referencia: persistencia,0.904,0.789,0.878,NaN,NaN,0.718
Regresión logística,0.997,0.659,0.904,0.817,0.200,0.933
Bosque aleatorio,0.995,0.685,0.913,0.840,0.400,0.933
Gradient boosting,1.000,0.647,0.902,0.839,0.100,0.931


La **persistencia** (dentro de dos años estará como hoy) es una referencia exigente porque el estado legal tiene mucha inercia: el criterio de precios mira cinco años hacia atrás. Lo que aporta el modelo es detectar los cambios de estado, que la persistencia por definición nunca anticipa.

In [9]:
pd.DataFrame(modelo['provincias']['importancias'])[['descripcion', 'importancia', 'desviacion']].head(8)

,descripcion,importancia,desviacion
0,Nivel de precio (log €/m²),0.046,0.011
1,Tensionado hoy (0/1),0.022,0.004
2,Precio − IPC a cinco años (puntos),0.019,0.004
3,Distancia al umbral del 30 % (puntos),0.011,0.004
4,Crecimiento de la renta en un año (%),0.010,0.002
5,Esfuerzo actual (cuota / renta),0.008,0.003
6,Crecimiento del alquiler en el año (%),0.006,0.004
7,Esfuerzo de alquiler (alquiler + gastos / renta),0.001,0.002


In [10]:
pd.DataFrame(modelo['provincias']['coeficientes'])[['descripcion', 'coeficiente']].head(8)

,descripcion,coeficiente
0,Tipo de interés hipotecario (%),1.867
1,Nivel de precio (log €/m²),1.802
2,Precio − IPC a cinco años (puntos),1.133
3,Crecimiento del alquiler en el año (%),1.060
4,Cambio del tipo en un año (puntos),-0.613
5,Crecimiento del precio en un año (%),0.573
6,Crecimiento de la renta en un año (%),-0.561
7,Tensionado hoy (0/1),0.523


## 5. Previsión a dos años

In [11]:
pred = pd.read_parquet(PUB / 'predicciones_provincias.parquet')
pred['escenario'].value_counts()

escenario
Seguirá tensionado      35
Seguirá sin tensión     11
Saldrá de la tensión     6
Name: count, dtype: int64

In [12]:
pred[['provincia', 'probabilidad', 'escenario', 'esfuerzo', 'diferencial']].head(12)

,provincia,probabilidad,escenario,esfuerzo,diferencial
0,Málaga,0.970,Seguirá tensionado,0.361,41.718
1,"Balears, Illes",0.960,Seguirá tensionado,0.346,39.188
2,Madrid,0.958,Seguirá tensionado,0.337,31.809
3,Barcelona,0.944,Seguirá tensionado,0.301,12.046
4,Gipuzkoa,0.885,Seguirá tensionado,0.308,5.481
5,Santa Cruz de Tenerife,0.849,Seguirá tensionado,0.256,38.695
6,"Palmas, Las",0.846,Seguirá tensionado,0.243,17.536
7,Segovia,0.842,Seguirá tensionado,0.209,35.593
8,Burgos,0.832,Seguirá tensionado,0.204,12.355
9,"Rioja, La",0.831,Seguirá tensionado,0.203,13.290
